In [ ]:
# Test a wide range of alphas across 3 orders of magnitude
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
ridge_results = []

for a in alphas:
    # Create and fit pipeline
    model = make_pipeline(
        OneHotEncoder(use_cat_names=True),
        StandardScaler(),
        Ridge(alpha=a)  # <--- Ridge(alpha=a)
    )
    model.fit(X_train_3, y_train_3) # <-- fit on train

    # Calculate RMSE for training and test
    rmse_train = root_mean_squared_error(y_train_3, model.predict(X_train_3))  # <-- y_train_3 & X_train_3
    rmse_test = root_mean_squared_error(y_test_3, model.predict(X_test_3))   # <-- y_test_3 & X_test_3

    ridge_results.append({
        "alpha": a,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test
    })

df_ridge_results = pd.DataFrame(ridge_results)
print(df_ridge_results)

      alpha    rmse_train     rmse_test
0     0.001  34200.432952  33088.372087
1     0.010  34200.432953  33088.367313
2     0.100  34200.432974  33088.319603
3     1.000  34200.435086  33087.845694
4    10.000  34200.639955  33083.415038
5   100.000  34216.977947  33062.388861
6  1000.000  35069.439956  33744.376028


**Code 2.3.5.2**: Create a line plot showing the U-shaped bias-variance tradeoff for Ridge. Plot training RMSE and test RMSE on the y-axis, with alpha on the x-axis (log scale).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(df_ridge_results["alpha"], df_ridge_results["rmse_train"],
        marker='o', linewidth=2, markersize=8, label="Training RMSE (Low Bias)")
ax.plot(df_ridge_results["alpha"], df_ridge_results["rmse_test"],
        marker='s', linewidth=2, markersize=8, label="Test RMSE (Generalization)")
ax.axvline(x=1, color='red', linestyle='--', alpha=0.5, label="α=1 (Our choice)")
ax.set_xlabel("alpha (Regularization Strength)", fontsize=12)
ax.set_ylabel("RMSE ($)", fontsize=12)
ax.set_xscale("log")
ax.set_title("Ridge Regression: The Bias-Variance Tradeoff", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Interpreting the U-Shape**
>
> The plot reveals the bias-variance tradeoff in action:
>
> - **Left side (low $\alpha$ like 0.001):** Training RMSE is low but test RMSE is substantially higher. The model is overfitting — it has memorized the training set but generalizes poorly. The large train-test gap is the diagnostic signature of overfitting.
>
> - **Bottom of the U ($\alpha \approx 1$):** Test RMSE is minimized. This is the "sweet spot" — the model balances complexity and generalizability. Neither under- nor over-regularized.
>
> - **Right side (high $\alpha$ like 1000):** Both training and test RMSE are high and nearly equal. The model is underfitting — the regularization is so strong that important features are being suppressed. The model has lost the ability to capture real patterns.
>
> **The diagnostic rule:** If train RMSE ≪ test RMSE → increase $\alpha$ (reduce variance). If both are high and similar → decrease $\alpha$ (reduce bias). The U-curve tells you which direction to move.

**Now it is time to answer MCQ 2.3.5.1.**

### 5.2 Lasso: Feature Selection with Regularization

Lasso behaves similarly to Ridge at extreme $\alpha$ values, but the middle of the U-curve has a different character: at intermediate $\alpha$ values, Lasso is simultaneously reducing overfitting *and* selecting features. The number of non-zero coefficients decreases monotonically as $\alpha$ increases.

**Code Task 2.3.5.3**: Create a Lasso tuning loop testing alphas `[0.1, 1, 10, 100, 1000]`.

In [ ]:
lasso_results = []
alphas_lasso = [0.1, 1, 10, 100, 1000]

for a in alphas_lasso:
    model = make_pipeline(
        OneHotEncoder(use_cat_names=True),
        StandardScaler(),
        Lasso(alpha=a, max_iter=10000)
    )
    model.fit(X_train_3, y_train_3)  # <-- fit on train

    rmse_train = root_mean_squared_error(y_train_3, model.predict(X_train_3))  # <-- y_train_3, X_train_3
    rmse_test = root_mean_squared_error(y_test_3, model.predict(X_test_3))  # <-- y_test_3, X_test_3
    non_zero_features = (model.named_steps["lasso"].coef_ != 0).sum()

    lasso_results.append({
        "alpha": a,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "non_zero_features": non_zero_features
    })

df_lasso_results = pd.DataFrame(lasso_results)
print(df_lasso_results)

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.670e+11, tolerance: 1.781e+09
  model = cd_fast.enet_coordinate_descent(


    alpha    rmse_train     rmse_test  non_zero_features
0     0.1  34200.432980  33088.297202                 59
1     1.0  34200.435537  33087.686878                 58
2    10.0  34200.691205  33081.797922                 58
3   100.0  34224.825010  33053.242280                 56
4  1000.0  34772.392675  33646.034886                 27


**Code 2.3.5.4**: Compare Ridge and Lasso alpha selection visually.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 6))

# Ridge comparison
ax1.plot(df_ridge_results["alpha"], df_ridge_results["rmse_test"],
         marker='o', linewidth=2, markersize=8, label="Ridge Test RMSE")
ax1.axvline(x=1, color='red', linestyle='--', alpha=0.7, linewidth=2, label="α=1 (Ridge default)")
ax1.set_xlabel("alpha (Regularization Strength)")
ax1.set_ylabel("Test RMSE ($)")
ax1.set_xscale("log")
ax1.set_title("Ridge: Alpha vs Test RMSE")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Lasso comparison with feature count
ax2_twin = ax2.twinx()
ax2.plot(df_lasso_results["alpha"], df_lasso_results["rmse_test"],
         marker='s', linewidth=2, markersize=8, color='green', label="Lasso Test RMSE")
ax2_twin.plot(df_lasso_results["alpha"], df_lasso_results["non_zero_features"],
              marker='^', linewidth=2, markersize=8, color='orange', linestyle='--', label="Non-zero Features")
ax2.axvline(x=100, color='red', linestyle='--', alpha=0.7, linewidth=2, label="α=100 (Our choice)")
ax2.set_xlabel("alpha (Regularization Strength)")
ax2.set_ylabel("Test RMSE ($)", color='green')
ax2_twin.set_ylabel("Number of Non-zero Features", color='orange')
ax2.set_xscale("log")
ax2.set_title("Lasso: Alpha vs Test RMSE & Feature Count")
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> **Why $\alpha = 100$ for Lasso?**
>
> From the tuning loop results, you can see distinct behavior across the range:
>
> - **Low $\alpha$ (0.1):** Lasso keeps most features active. Performance is good but the model may be retaining noise features. The model is closer to ordinary linear regression.
> - **$\alpha = 100$:** A good balance — aggressive enough to eliminate many noise features (sparse model) while keeping test RMSE reasonable. This is the "feature selection regime" where Lasso's unique strength is most visible.
> - **High $\alpha$ (1000):** Lasso has driven so many coefficients to zero that even useful features are being eliminated. Test RMSE rises — underfitting.
>
> $\alpha = 100$ is in the regime where Lasso performs meaningful feature selection without sacrificing too much predictive accuracy. However, the optimal $\alpha$ depends on your specific dataset — which is precisely why we use **LassoCV** to find it automatically.

### 5.3 Automated Alpha Selection: RidgeCV and LassoCV

Manually testing a grid of $\alpha$ values and eyeballing the U-curve is informative but not rigorous. **Cross-validation** provides a principled, statistically reliable method for selecting $\alpha$.

> **What is cross-validation?**
>
> Instead of a single train-test split, cross-validation divides the training data into $k$ folds (typically 5 or 10). For each candidate $\alpha$:
>
> 1. Train on $k-1$ folds, evaluate on the held-out fold
> 2. Repeat $k$ times (each fold takes a turn as the held-out set)
> 3. Average the $k$ error estimates
> 4. Select the $\alpha$ with the lowest average cross-validation error
>
> This approach uses all available training data for both training and validation, producing a more reliable estimate of generalization performance than a single split. A single train-test split estimate of performance has high variance — two different splits of the same data can yield quite different "optimal" alphas. Cross-validation averages out this variance by using all possible held-out folds.

`RidgeCV` and `LassoCV` implement this procedure automatically — you simply pass a list of candidate alphas and the model selects the best one internally. The selected alpha is stored in `model.named_steps["ridgecv"].alpha_` after fitting.

**Code 2.3.5.5**

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

# RidgeCV automatically finds the best alpha
ridge_cv_model = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    StandardScaler(),
    RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100, 1000], cv=5)  # <--- 5-fold cross-validation
)
ridge_cv_model.fit(X_train_3, y_train_3)

# Extract the optimal alpha
optimal_alpha_ridge = ridge_cv_model.named_steps["ridgecv"].alpha_
print(f"RidgeCV selected alpha: {optimal_alpha_ridge}")
print(f"Ridge CV Test RMSE: ${root_mean_squared_error(y_test_3, ridge_cv_model.predict(X_test_3)):,.2f}")

# LassoCV automatically finds the best alpha
lasso_cv_model = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    StandardScaler(),
    LassoCV(alphas=[0.1, 1, 10, 100, 1000], cv=5, max_iter=10000)
)
lasso_cv_model.fit(X_train_3, y_train_3)

optimal_alpha_lasso = lasso_cv_model.named_steps["lassocv"].alpha_
print(f"\nLassoCV selected alpha: {optimal_alpha_lasso}")
print(f"Lasso CV Test RMSE: ${root_mean_squared_error(y_test_3, lasso_cv_model.predict(X_test_3)):,.2f}")
print(f"Non-zero features (Lasso CV): {(lasso_cv_model.named_steps['lassocv'].coef_ != 0).sum()}")

> **RidgeCV and LassoCV — Key Properties**
>
> - `RidgeCV(alphas=[...])` automatically selects the best $\alpha$ from the provided list using cross-validation. The chosen alpha is stored in `model.alpha_` after fitting.
> - `LassoCV(cv=5)` similarly selects the optimal alpha and stores it in `model.alpha_`. The `cv` parameter specifies the number of folds.
> - Both integrate naturally into Pipelines: `make_pipeline(OHE, Scaler, RidgeCV(alphas=[...]))` works exactly like the non-CV versions.
>
> **Why not always use CV?** Cross-validation is computationally more expensive — it fits the model $k \times |\text{alphas}|$ times instead of once. For small datasets and fast models (like Ridge and Lasso), this cost is negligible. For large datasets or slow models, you might accept a slightly suboptimal $\alpha$ to save computation.

---

## 6. Visualizing and Interpreting Coefficients

The true power of Lasso's feature selection becomes visible when we examine the coefficients directly. Let's extract them and see which features the model kept and which it zeroed out.

**Code Task 2.3.6.1**: Extract the coefficients and feature names from `model_lasso_3`. Create a Series named `feat_imp_3`.

In [ ]:
# Extract coefficients from the Lasso step
coefficients = model_lasso_3.named_steps["lasso"].coef_  # <--- .coef_

# Extract feature names from the OneHotEncoder step
features = model_lasso_3.named_steps["onehotencoder"].get_feature_names_out()

# Create Series
feat_imp_3 = pd.Series(coefficients, index=features)  # <--- pd.Series(coefficients, index=features)

print(f"Total features: {len(feat_imp_3)}")
print(f"Non-zero features: {(feat_imp_3 != 0).sum()}")

Total features: 59
Non-zero features: 56


> **Sparsity as a Competitive Advantage**
>
> Observe the dramatic feature compression Lasso achieved:
>
> - **Before encoding:** ~5 raw columns (surface area, lat, lon, neighborhood as one column)
> - **After OneHotEncoder:** ~53 features (neighborhood alone expands to 50+ binary columns)
> - **After Lasso ($\alpha = 100$):** ~20 features with non-zero coefficients — roughly 62% of features zeroed out
>
> This is **automatic feature selection** in action. Lasso identified which of the 50+ neighborhood columns genuinely predict price and which are noise, eliminating the noise without human intervention.
>
> Why does this matter?
>
> 1. **Interpretability:** A model with 20 non-zero features is far easier to explain than one with 53. You can enumerate which neighborhoods matter and quantify their price effect.
> 2. **Computational efficiency:** Fewer active features mean faster predictions in production.
> 3. **Generalization:** By eliminating features with insufficient signal, Lasso reduces the chance of overfitting to training-set peculiarities.
>
> This feature-selection property is Lasso's defining advantage over Ridge for high-dimensional problems where many features are suspected to be noise.

Now let's visualize the top coefficients.

**Code 2.3.6.2**: Plot the top 10 features by absolute coefficient value.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
feat_imp_3.sort_values(key=abs).tail(10).plot(kind="barh", ax=ax)
ax.set_title("Top 10 Coefficients: Lasso (alpha=100)")
ax.set_xlabel("Coefficient Value")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

**Now it is time to answer MCQ 2.3.6.1.**

**Code Task 2.3.6.3**: Count how many features have been set to exactly zero by the Lasso model.

In [ ]:
lasso_zeros_count = (feat_imp_3 == 0).sum()  # <--- (feat_imp_3 == 0).sum()
print(f"Lasso removed {lasso_zeros_count} features out of {len(feat_imp_3)} total features.")
print(f"Percentage removed: {lasso_zeros_count / len(feat_imp_3) * 100:.1f}%")

Lasso removed 3 features out of 59 total features.
Percentage removed: 5.1%


**Verify 2.3.6.3**

In [ ]:
assert 'lasso_zeros_count' in globals(), "❌ 'lasso_zeros_count' is not defined"
assert isinstance(lasso_zeros_count, (int, np.integer)), "❌ Should be an integer count"
assert lasso_zeros_count >= 0, "❌ Zero count should be non-negative"
assert lasso_zeros_count < len(feat_imp_3), "❌ Should not eliminate all features"
assert lasso_zeros_count + (feat_imp_3 != 0).sum() == len(feat_imp_3), "❌ Zero and non-zero counts should sum to total"
print("✓ All assertions passed")

**Now it is time to answer MCQ 2.3.6.2.**

### 6.4 Interpreting Lasso Coefficients

Lasso coefficients require more careful interpretation than OLS coefficients. Two important caveats apply:

**Caveat 1: Coefficients are on the standardized scale**

After `StandardScaler`, every feature has mean = 0 and standard deviation = 1. A coefficient of +50,000 on `neighborhood_Palermo` does not mean "Palermo adds $50,000 to the price in raw square-meter terms." It means Palermo adds $50,000 per *standard deviation* of the scaled feature — which, for a binary indicator column, essentially means $50,000 per unit of being in Palermo.

For binary indicator columns (all our neighborhood columns), the standardized and raw interpretations coincide closely: a coefficient of +$X means being in that neighborhood is associated with approximately +$X relative to the baseline neighborhood, after accounting for all other features.

**Caveat 2: Coefficients reflect the post-selection model**

Lasso has already eliminated ~60% of features. The surviving coefficients are not just estimates of those features' effects in isolation — they are estimates *after the other eliminated features have been removed*. If Lasso zeroed out `neighborhood_Recoleta` but kept `neighborhood_Palermo`, the Palermo coefficient absorbs some of the effect that Recoleta might have partially shared.

> **Practical interpretation guideline:**
>
> For business communication, the cleanest interpretation is relative: "Apartments in neighborhood X are predicted to sell for approximately $Y more (or less) than apartments in the baseline neighborhood, holding size and location fixed." The exact dollar amount is an approximation, but the direction and rough magnitude are reliable.

**Code 2.3.6.4**: Display the top 5 positive and negative features from the fitted Lasso model.

In [ ]:
# Show top 5 positive and negative coefficients
print("=" * 60)
print("TOP 5 POSITIVE FEATURES (Increase Price)")
print("=" * 60)
top_positive = feat_imp_3.nlargest(5)
for feature, coef in top_positive.items():
    print(f"{feature:40} : ${coef:>10,.2f}")

print("\n" + "=" * 60)
print("TOP 5 NEGATIVE FEATURES (Decrease Price)")
print("=" * 60)
top_negative = feat_imp_3.nsmallest(5)
for feature, coef in top_negative.items():
    print(f"{feature:40} : ${coef:>10,.2f}")

print("\n" + "=" * 60)
print("INTERPRETATION GUIDE")
print("=" * 60)
print("Remember: These coefficients are on STANDARDIZED features.")
print("A feature's importance depends on both its coefficient AND")
print("which other features Lasso eliminated. This is the 'feature selection'")
print("effect of Lasso in action.")

### 6.5 Residual Analysis: Univariate vs. Multivariate

Adding neighborhood, latitude, and longitude to the model should capture more of the systematic price variation that size alone could not explain. Let's verify this by comparing residual patterns.

**Hypothesis:** The multivariate residuals should be more randomly scattered than the univariate residuals from Lesson 2, because the model now captures neighborhood-level price effects that were previously "missing" and showing up as systematic patterns.

**Reading the residual plot for a multivariate model:**

With a single feature, we could plot residuals against that feature and directly see the pattern. With multiple features (surface area + lat + lon + 50+ neighborhood indicators), we plot residuals against the **predicted values** instead — the only single axis that summarizes all feature contributions.

The same interpretive rules apply:
- **Random scatter around zero:** All systematic variation is captured; remaining errors are unpredictable noise.
- **Funnel shape (heteroscedasticity):** Error variance increases with predicted price — a common pattern in financial data that linear models struggle to fully resolve.
- **Systematic curve:** The relationship is non-linear in ways the model's current features cannot capture.

**Code 2.3.6.5**: Create a residual plot for the multivariate Lasso model and compare it to the univariate model's residuals.

In [ ]:
# Get residuals from Lasso predictions on test set
y_pred_lasso = model_lasso_3.predict(X_test_3)
residuals_lasso = y_test_3 - y_pred_lasso

# Create residual plot
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(y_pred_lasso, residuals_lasso, alpha=0.5, s=30)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2, label='Zero Error')
ax.set_xlabel("Predicted Price (USD)", fontsize=12)
ax.set_ylabel("Residuals (USD)", fontsize=12)
ax.set_title("Lasso (Multivariate) Model: Residual Plot", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics
print(f"Mean of residuals: ${residuals_lasso.mean():,.2f} (should be close to 0)")
print(f"Std of residuals: ${residuals_lasso.std():,.2f}")
print(f"Min residual: ${residuals_lasso.min():,.2f}")
print(f"Max residual: ${residuals_lasso.max():,.2f}")

> **Comparing Univariate (Lesson 2) vs. Multivariate (Lesson 3) Residuals**
>
> **Univariate model (Lesson 2) — one feature: surface area**
> - Residuals showed systematic funnel-shaped heteroscedasticity — variance increased at higher prices
> - Systematic underestimation for some neighborhoods, overestimation for others
> - The model could not distinguish a 50m² apartment in Palermo from a 50m² apartment in a peripheral neighborhood
>
> **Multivariate model (Lesson 3) — surface area + coordinates + neighborhood**
> - Residuals should be more randomly scattered — less systematic structure
> - The model now differentiates apartments by location, not just size
> - The funnel shape may still be present (heteroscedasticity is a real property of price data) but should be less severe
>
> ❗️ **If the multivariate residuals still show systematic patterns**, it means important relationships remain uncaptured — non-linearities in the size-price relationship, interactions between neighborhood and size, or other factors our feature set does not include. Residual analysis is a diagnostic tool: it tells you what your current model is missing.

**Code 2.3.7.1**: Build a Ridge vs. Lasso comparison table showing training and test RMSE for both models.

In [ ]:
ridge_metrics = {
    "Model": "Ridge (α=1.0)",
    "RMSE_train": rmse_ridge_train,
    "RMSE_test": rmse_ridge_test,
    "Non-zero_features": "All"
}

lasso_metrics = {
    "Model": "Lasso (α=100)",
    "RMSE_train": root_mean_squared_error(y_train_3, model_lasso_3.predict(X_train_3)),
    "RMSE_test": rmse_lasso_test,
    "Non-zero_features": (feat_imp_3 != 0).sum()
}

df_comparison = pd.DataFrame([ridge_metrics, lasso_metrics])
print(df_comparison)

---

## Summary

In this lesson, you moved from a single-feature linear model to regularized multi-feature models that can handle high-dimensional data without overfitting. Here is what each step built on the previous:

| Step | What we did | Why it matters |
|------|-------------|----------------|
| **Feature setup** | Added neighborhood (50+ categories), lat, lon to X | More signals → better potential accuracy, but also more overfitting risk |
| **StandardScaler** | Transformed all features to mean=0, std=1 | Ensures the regularization penalty treats all features fairly |
| **Pipeline** | Chained OHE → Scaler → Model in one object | Prevents data leakage from transformers fitted on test data |
| **Ridge (L2)** | Penalty $\alpha \sum \beta_j^2$ — shrinks all coefficients | Reduces variance without removing any features |
| **Lasso (L1)** | Penalty $\alpha \sum \|\beta_j\|$ — zeroes out weak features | Automatic feature selection; sparse, interpretable model |
| **Alpha tuning** | Tested range of alphas; plotted U-curve | Found the bias-variance tradeoff visually |
| **RidgeCV / LassoCV** | Cross-validation for automatic alpha selection | Principled, reproducible hyperparameter selection |
| **Coefficient analysis** | Extracted non-zero features from Lasso | Identified which neighborhoods drive price most strongly |
| **Residual comparison** | Compared L2 (univariate) vs L3 (multivariate) residuals | Verified that adding features reduced systematic errors |

**Key Takeaways:**

- **Feature scaling is mandatory for regularization** — without it, the penalty unfairly targets features with different units or ranges
- **Pipelines are architecturally correct** — they are not just convenience; they are the only way to guarantee that transformers are fit only on training data
- **Ridge keeps all features; Lasso selects features** — choose based on whether you believe most features contribute (Ridge) or only some (Lasso)
- **Alpha is the complexity dial** — low alpha ≈ ordinary regression (high variance risk), high alpha ≈ constant prediction (high bias risk), optimal alpha is in the U-curve's minimum
- **Lasso coefficients are sparse** — the zero coefficients are the model's statement that those features are not worth including at this alpha level
- **Coefficient interpretation requires care** — Lasso coefficients are on the standardized scale and reflect the post-selection model

**Now it is time to answer MCQ 2.3.7.1.**

---

## Discussion Questions

1. **Ridge vs. Lasso:** Given that our dataset has 50+ neighborhood columns and we believe only a subset of neighborhoods significantly affect price, which regularization method is more appropriate? Why?

2. **Pipeline order:** We applied `StandardScaler` *after* `OneHotEncoder`. Why does this order make sense? What would go wrong if we scaled before encoding?

3. **Alpha tuning:** If your model's training RMSE is much lower than its test RMSE, which direction should you move $\alpha$? Explain why in terms of the bias-variance tradeoff.

4. **Coefficient interpretation:** Look at the top 10 features from your Lasso model. Which neighborhoods have the strongest positive and negative price associations? Do these make intuitive sense given what you know about Buenos Aires?

5. **Residuals:** After adding neighborhood and location features, did the residual plots improve compared to the single-feature model? If systematic patterns remain, what additional features might help?